# Colab 1 · AutoGen AgentChat Fundamentals
### Day 19 — Agent Orchestration with AutoGen Studio & Semantic Kernel

In this notebook you'll go from **one agent** to a **two-agent team that delegates and self-reviews** — the smallest interesting multi-agent system.

**You will:**
1. Install AutoGen AgentChat and connect a model client.
2. Build a single `AssistantAgent` and run it.
3. Compose a **writer + critic** team with `RoundRobinGroupChat`.
4. Add **termination conditions** so the team stops cleanly.
5. (Optional) Launch **AutoGen Studio** and export your team as JSON.

> AutoGen is in *maintenance mode* in 2026 and the concepts here map directly onto Semantic Kernel and the Microsoft Agent Framework. Learn the ideas once; the SDK is a detail.

⏱️ ~45 min including the extension tasks at the bottom.


## 0 · Setup

Run the install cell, then paste your OpenAI API key when prompted. (You can also adapt the model client to Azure OpenAI or a local model — the rest of the notebook is unchanged.)

In [ ]:
%pip install -q -U "autogen-agentchat" "autogen-ext[openai]"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Obtaining dependency information for groq from https://files.pythonhosted.org/packages/ce/c3/cac2aee75198c0382a2e6070619a798de2288e01850abb6e999746bb92e3/groq-1.5.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/143.7 kB ? eta -:--:--
   -- ------------------------------------- 10.2/143.7 kB ? eta -:--:--
   ---------------------------------------  143.4/143.7 kB 2.1 MB/s eta 0:00:01
   ---------------------------------------  143.4/143.7 kB 2.1 MB/s eta 0:00:01
   ---------------------------------------  143.4/143.7 kB 2.1 MB/s eta 0:00:01
   -------------------------------------- 143.7/143.7 kB 709.8 kB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 0.37.1
    Uninstalling groq-0.37.1:
      Successfully uninstalled groq-0.37.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.3 requires groq<1.0.0,>=0.30.0, but you have groq 1.5.0 which is incompatible.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os

# Hard-coded API key (for local development only)
os.environ["OPENAI_API_KEY"] = "openai_api_key_goes_here"

print("Key set:", bool(os.environ.get("OPENAI_API_KEY")))

Key set: True


## 1 · The model client

Every agent needs a **model client** — the thing that actually calls the LLM. We use a cheap, fast model for the workshop. The client reads `OPENAI_API_KEY` from the environment automatically.

In [14]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini"
)

print(model_client)

## 2 · Your first agent

An `AssistantAgent` is a name + a model + a `system_message` that fixes its role. The `description` matters later: in **Selector** and **Swarm** teams, other agents route work based on it.

In [15]:
from autogen_agentchat.agents import AssistantAgent

writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    description="Writes short marketing copy.",
    system_message="You are a helpful assistant."
)

In [16]:
from autogen_agentchat.agents import AssistantAgent

writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    system_message="You are a helpful assistant."
)

In [17]:
result = await writer.run(
    task="Write 5 lines about Artificial Intelligence."
)

print(result.messages[-1].content)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************U50A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

## 3 · A two-agent team: writer + critic

One agent can't reliably critique its own work. Let's **delegate the review** to a second agent and let them take turns with `RoundRobinGroupChat` — the simplest orchestration pattern: a **fixed turn order, shared context, no routing logic**.

The critic's job is to either suggest improvements *or* approve. We'll use the word `APPROVE` as our stop signal in the next step.

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

critic = AssistantAgent(
    name="critic",
    model_client=model_client,
    description="Reviews copy and gives specific, actionable feedback.",
    system_message=(
        "You are a sharp editor. Critique the writer's paragraph in 2-3 bullet points. "
        "If it is genuinely strong, reply with exactly the word APPROVE and nothing else."
    ),
)

### Termination: always cap the run

A multi-agent team will loop forever if you let it. We combine **two** conditions with `|` (OR — stop when *either* fires):

* `TextMentionTermination("APPROVE")` — stop when the critic signs off.
* `MaxMessageTermination(6)` — a hard safety cap so we never loop or burn tokens endlessly.

This pattern — *a stop word **and** a hard cap* — is the single most important habit in this workshop.

In [ ]:
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(6)

team = RoundRobinGroupChat(
    participants=[writer, critic],
    termination_condition=termination,
)
print("Team built with", len(team._participants) if hasattr(team, "_participants") else 2, "agents")

### Run it and watch the delegation happen

`run_stream` yields each message as it's produced; wrapping it in `Console` pretty-prints the whole conversation so you can *see* the hand-off between writer and critic.

In [ ]:
from autogen_agentchat.ui import Console

await team.reset()  # clear any state from earlier runs
await Console(
    team.run_stream(task="Write a blurb for a reusable, dishwasher-safe coffee cup.")
)

## 4 · Read the conversation

Look at the transcript above and notice:

* The **writer** drafts, the **critic** responds — that's task delegation in action.
* The run **stopped on its own** — either the critic said `APPROVE`, or we hit the 6-message cap.
* All agents shared **one conversation context**; that's what `RoundRobinGroupChat` gives you for free.

You can also get the result without streaming:

In [ ]:
await team.reset()
result = await team.run(task="Write a blurb for noise-cancelling headphones for students.")
print("Messages exchanged:", len(result.messages))
print("Stop reason:", result.stop_reason)
print("\nFinal message:\n", result.messages[-1].content)

## 5 · (Optional) See the same team in AutoGen Studio

Everything above can be built **with no code** in AutoGen Studio's Team Builder, and any Studio team **exports to JSON** that you can load back into code. The GUI and AgentChat are the same engine.

Running Studio inside Colab is fiddly (it wants its own web server); the cell below shows the commands you'd run **locally**:

```bash
pip install -U autogenstudio
autogenstudio ui --port 8080 --appdir ./my-app
# then open http://localhost:8080
```

In Studio you would: add two agents, set their descriptions, drop them into a RoundRobin team, set the termination, and hit **Run** in the Playground. Then **export** the team to a `.json` component config and load it with `Team.load_component(config)`.

> ⚠️ Studio has no built-in auth or jailbreak hardening — it's a prototyping tool. Never expose it raw to the internet, and run code-executing agents inside Docker.

---
## 🚀 Extension tasks

Work through as many as time allows. Each builds directly on the team above.

### Extension 1 — Add a third agent (an *editor*)
Insert an `editor` agent **after** the critic that produces the final, polished version and ends its message with `APPROVE`. Change the critic so it only critiques (never approves), and let the editor own the stop word. You now have a **3-stage delegation pipeline**: writer → critic → editor.

### Extension 2 — Right-size the model
Give the **critic** a cheaper/faster model and keep a stronger model only where it matters. Create a second `OpenAIChatCompletionClient` (e.g. a smaller model) and pass it to the critic. This is the *cost-control* habit from the slides: cheap model for routine agents, strong model for the hard one.

### Extension 3 — Give the writer a tool
Define a plain Python function `count_words(text: str) -> int` and pass it to the **writer** via `tools=[count_words]`. Update the writer's system message to *call the tool and keep the blurb under 40 words*. Watch the tool call appear in the streamed transcript — that's a delegated function call.

The scaffold cells below give you a starting point.

In [ ]:
# === Extension 1 scaffold: writer -> critic -> editor ===
# TODO: create an `editor` AssistantAgent that outputs the final copy and ends with APPROVE.
# TODO: change `critic` so it never says APPROVE (critique only).
# TODO: build a RoundRobinGroupChat([writer, critic, editor]) and run it.

editor = AssistantAgent(
    name="editor",
    model_client=model_client,
    description="Produces the final, polished version of the copy.",
    system_message=(
        "You are the final editor. Incorporate the critic's feedback, output the final "
        "polished paragraph, then on a new line write exactly APPROVE."
    ),
)

# TODO: rebuild the team with three agents and run it.
# pipeline = RoundRobinGroupChat([writer, critic, editor], termination_condition=termination)
# await Console(pipeline.run_stream(task="Write a blurb for a bamboo toothbrush."))

In [ ]:
# === Extension 2 scaffold: a cheaper model for the critic ===
# cheap_client = OpenAIChatCompletionClient(model="gpt-4o-mini")   # or an even smaller model
# critic = AssistantAgent(name="critic", model_client=cheap_client, description="...", system_message="...")
# Rebuild the team and confirm it still works.

In [ ]:
# === Extension 3 scaffold: a word-count tool for the writer ===
def count_words(text: str) -> int:
    """Return the number of whitespace-separated words in `text`."""
    return len(text.split())

# TODO: pass tools=[count_words] when constructing the writer, and update its
# system_message to call the tool and keep the blurb under 40 words.
# writer_with_tool = AssistantAgent(
#     name="writer", model_client=model_client, tools=[count_words],
#     description="Writes copy and checks length with a tool.",
#     system_message="Write the blurb, call count_words on it, and ensure it is under 40 words.",
# )

## Recap

You built a real multi-agent system: two agents **delegating** drafting and review, orchestrated with **RoundRobin**, stopped by a **combined termination condition**. You saw how the GUI (Studio) and code (AgentChat) are the same thing, and you extended the team with a third role, a cheaper model, and a tool.

**Next:** Colab 2 takes you beyond round-robin into **Selector, Swarm, GraphFlow** and a peek at **Semantic Kernel** orchestration.

Don't forget to close the client when you're done:

In [ ]:
await model_client.close()
print("Model client closed. Nice work!")